# 05 — Build Feature Table
# Giai đoạn 1 — Xây dựng bảng 28 đặc trưng cho các phương pháp envelope

**Mục tiêu:** tạo ba bảng đặc trưng có cùng tập cửa sổ và cùng 28 cột feature, chỉ khác nhóm đặc trưng envelope:

1. `hybrid`: phương pháp đề xuất.
2. `hilbert`: FFT Hilbert toàn file, tham chiếu độ trung thực trên PC (không dùng làm đối chứng chi phí MCU).
3. `hilbert_block`: FFT Hilbert theo block 2048 mẫu, mô phỏng đúng cấu hình xử lý dự kiến port sang MCU. Theo `dsp.py` hiện tại, block cuối chưa đủ 2048 mẫu được FFT theo độ dài thực còn lại; phải đồng bộ chính sách này với firmware C trước khi kiểm tra PC–MCU.

**Đầu ra:**
- `outputs/tables/features_mlp_hybrid.parquet`
- `outputs/tables/features_mlp_hilbert.parquet`
- `outputs/tables/features_mlp_hilbert_block.parquet`
- `outputs/tables/features_mlp.parquet` (alias của Hybrid)
- Các file CSV tương ứng để kiểm tra thủ công.
- `outputs/tables/manifest_with_folds.csv` và thông tin SHA-256 do module splitting xuất.

**Nguyên tắc:** mỗi file được lọc envelope một lần trên toàn tín hiệu trước khi cắt cửa sổ; ba nhánh phải dùng cùng manifest, RPM, cửa sổ, stride, warm-up và cấu hình BPF/LPF.


In [1]:
from pathlib import Path
import importlib
import pandas as pd
import json
import numpy as np
from src import io_utils, features_full, splitting, config as cfg, dsp

# Reload DSP trước; features_full giữ tham chiếu đến cùng module dsp.
importlib.reload(dsp)
importlib.reload(features_full)


<module 'src.features_full' from 'F:\\CODE\\NCKH_TinyML\\TinyML_CWRU\\src\\features_full.py'>

In [2]:
OUTPUT_DIR = Path("./outputs")
TABLES_DIR = OUTPUT_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Đọc manifest và cấu hình


In [3]:
# Tạo manifest sau scope-filter và gán fold theo file.
# Việc lưu manifest ở đây giúp notebook 04/06/07 sử dụng đúng cùng một bản.
manifest_full = io_utils.build_manifest(Path("../../data/raw"))
manifest_clean = io_utils.apply_scope_filter(manifest_full)

manifest_clean, manifest_hash = splitting.save_manifest_with_folds(
    manifest_clean,
    TABLES_DIR / "manifest_with_folds.csv",
    seeds=range(42, 52),
)
manifest_clean.to_csv(TABLES_DIR / "manifest_clean.csv", index=False)
print(f"Manifest SHA-256: {manifest_hash}")

bandpass_config_path = TABLES_DIR / "bandpass_config.json"
if not bandpass_config_path.exists():
    raise FileNotFoundError(
        f"Không tìm thấy {bandpass_config_path}. Hãy chạy notebook chọn bandpass trước."
    )
with open(bandpass_config_path, encoding="utf-8") as f:
    bandpass_cfg = json.load(f)

required_manifest_cols = [
    "file_path",
    "label",
    "load_hp",
    "fault_diameter_mils",
    "warnings",
    "has_warning",
    "resolved_sample_rate_hz",
    "resolved_sample_rate_source",
    "rpm",
    "rpm_source",
]
missing = [column for column in required_manifest_cols if column not in manifest_clean.columns]
if missing:
    raise ValueError(f"manifest_clean thiếu các cột bắt buộc: {missing}")

if manifest_clean["resolved_sample_rate_hz"].isna().any():
    raise ValueError(
        "manifest_clean còn file thiếu sampling rate đã xác thực. "
        "Kiểm tra lại notebook tạo manifest."
    )
if manifest_clean["rpm"].isna().any() or manifest_clean["rpm_source"].isna().any():
    raise ValueError("Manifest có RPM hoặc rpm_source bị thiếu.")

blocked_pattern = (
    "VONG_BI_NTN|OR_NGOAI_PHAM_VI|OR_THIEU_VI_TRI|THIEU_NHAN|THIEU_TAI|"
    "DUONG_KINH_LA|NGOAI_PHAM_VI_CAM_BIEN|NGOAI_PHAM_VI_TAN_SO_KHAI_BAO|"
    "THIEU_TIN_HIEU_DE|LOI_DOC_FILE"
)
if manifest_clean["warnings"].fillna("").astype(str).str.contains(
    blocked_pattern, regex=True
).any():
    raise ValueError("manifest_clean vẫn chứa file vi phạm scope.")

BAND_HZ = tuple(float(v) for v in bandpass_cfg["band_hz"])
LP_CUTOFF_HZ = float(bandpass_cfg["lp_cutoff_hz"])
TARGET_FS = float(cfg.SCOPE["sampling_rate_hz"])
WINDOW_SIZE = int(cfg.WINDOW_SIZE_RAW)
STRIDE = int(cfg.STRIDE_RAW)
WARMUP = int(cfg.WARMUP_SAMPLES)
FS_COL = "resolved_sample_rate_hz"
BPF_ORDER = int(dsp.BENCHMARK_BANDPASS_ORDER)
LOWPASS_ORDER = int(dsp.BENCHMARK_LOWPASS_ORDER)
HILBERT_BLOCK_SIZE = int(getattr(dsp, "HILBERT_BLOCK_SIZE", WINDOW_SIZE))

# Không để notebook dùng một cấu hình khác với DSP đã khóa trong src/dsp.py.
if not np.allclose(BAND_HZ, dsp.BENCHMARK_BAND_HZ, rtol=0.0, atol=1e-9):
    raise ValueError(
        f"bandpass_config {BAND_HZ} không khớp dsp.BENCHMARK_BAND_HZ "
        f"{dsp.BENCHMARK_BAND_HZ}."
    )
if not np.isclose(LP_CUTOFF_HZ, dsp.BENCHMARK_LP_CUTOFF_HZ, rtol=0.0, atol=1e-9):
    raise ValueError(
        f"LP cutoff {LP_CUTOFF_HZ} không khớp dsp.BENCHMARK_LP_CUTOFF_HZ "
        f"{dsp.BENCHMARK_LP_CUTOFF_HZ}."
    )
if WINDOW_SIZE != HILBERT_BLOCK_SIZE:
    raise ValueError(
        f"WINDOW_SIZE={WINDOW_SIZE} phải trùng HILBERT_BLOCK_SIZE={HILBERT_BLOCK_SIZE}."
    )
if WINDOW_SIZE % dsp.ENV_DECIM or STRIDE % dsp.ENV_DECIM:
    raise ValueError("WINDOW_SIZE và STRIDE phải chia hết cho ENV_DECIM.")

print(f"Số file trong manifest: {len(manifest_clean)}")
print(f"Bandpass: {BAND_HZ} Hz, LP cutoff: {LP_CUTOFF_HZ} Hz")
print(
    f"Target FS: {TARGET_FS:g} Hz | Window: {WINDOW_SIZE} | "
    f"Stride: {STRIDE} | Warm-up: {WARMUP}"
)
print("Sampling rate nguồn:")
print(manifest_clean[FS_COL].value_counts().sort_index().to_string())
print("RPM source:")
print(manifest_clean["rpm_source"].value_counts(dropna=False).to_string())


Manifest SHA-256: 8b33c17604bb8ae6ce69fccac8fc123ffe908cc535aa3c2fccfe00ceeec2a4c4
Số file trong manifest: 40
Bandpass: (2300.0, 3800.0) Hz, LP cutoff: 750.0 Hz
Target FS: 12000 Hz | Window: 2048 | Stride: 1024 | Warm-up: 2048
Sampling rate nguồn:
resolved_sample_rate_hz
12000.0    36
48000.0     4
RPM source:
rpm_source
measured                   38
imputed:nominal_by_load     2


## 2. Hàm load + resample tường minh

Resample được thực hiện trong RAM khi đọc file. Metadata sampling rate nguồn được lấy từ manifest; không tự suy diễn lại trong bước build feature.


In [4]:
def load_and_resample(file_path, source_fs, target_fs=TARGET_FS):
    """Đọc tín hiệu DE và resample từ fs nguồn đã ghi trong manifest."""
    x = io_utils.load_de_signal_resampled(
        Path(file_path),
        float(source_fs),
        float(target_fs),
    )
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    if x.size == 0 or not np.isfinite(x).all():
        raise ValueError(f"Tín hiệu không hợp lệ sau resample: {file_path}")
    return x


# Kiểm tra nhanh một file Normal đại diện; toàn bộ file sẽ được nạp trong bước build.
normal_rows = manifest_clean.loc[manifest_clean["label"].eq("Normal")]
if normal_rows.empty:
    raise ValueError("Manifest không có file Normal để kiểm tra resample.")
for _, row in normal_rows.head(1).iterrows():
    x_test = load_and_resample(row["file_path"], row[FS_COL])
    print(
        f"{Path(row['file_path']).name}: "
        f"{float(row[FS_COL]):g} Hz -> {TARGET_FS:g} Hz | "
        f"{len(x_test)} mẫu | {len(x_test) / TARGET_FS:.2f} s"
    )


100_Normal_3.mat: 48000 Hz -> 12000 Hz | 121411 mẫu | 10.12 s


## 3. Hàm build bảng đặc trưng cho một DSP method

Với mỗi phương pháp, `features_full.build_full_feature_table()`:
1. đọc và resample tín hiệu theo manifest;
2. chạy BPF → phương pháp envelope → LPF/DC blocker trên toàn file;
3. decimate envelope ×2;
4. cắt cùng các cửa sổ raw/envelope theo `window_size`, `stride`, `warmup`;
5. trích 11 Time + 10 Order + 7 Envelope = **28 đặc trưng**.

Hilbert Block được cấu hình `float32=True`, `block_size=2048` ở phần FFT envelope. Các bộ lọc chung vẫn theo implementation của `src/dsp.py`.


In [5]:
# Cấu hình riêng theo phương pháp; hai nhánh còn lại không nhận kwargs bổ sung.
ENVELOPE_METHOD_KWARGS = {
    "hybrid": {},
    "hilbert_offline": {},
    "hilbert_block": {
        "block_size": HILBERT_BLOCK_SIZE,
        "float32": True,
        "align_delay": True,
    },
}


def build_features(method_canonical):
    """Build bảng feature 28 chiều cho toàn manifest với một DSP method."""
    if method_canonical not in dsp.BENCHMARK_ENVELOPE_METHODS:
        raise ValueError(
            f"Method không được hỗ trợ: {method_canonical}. "
            f"Chọn {dsp.BENCHMARK_ENVELOPE_METHODS}."
        )

    method_kwargs = ENVELOPE_METHOD_KWARGS.get(method_canonical, {})
    return features_full.build_full_feature_table(
        manifest_clean,
        band_hz=BAND_HZ,
        load_de_signal_fn=load_and_resample,
        window_size=WINDOW_SIZE,
        stride=STRIDE,
        warmup_samples=WARMUP,
        target_fs_hz=TARGET_FS,
        lp_cutoff_hz=LP_CUTOFF_HZ,
        bandpass_order=BPF_ORDER,
        lowpass_order=LOWPASS_ORDER,
        envelope_method=method_canonical,
        envelope_kwargs=method_kwargs,
    )


## 4. Build ba bảng đặc trưng cho ba DSP methods

Mỗi bảng giữ nguyên 28 feature và metadata. Chỉ nhóm `envelope_*` có thể khác nhau giữa các phương pháp.


In [6]:
# Tên nhánh đầu ra -> tên method chuẩn trong src/dsp.py
DSP_METHODS = {
    "hybrid": "hybrid",
    "hilbert": "hilbert_offline",
    "hilbert_block": "hilbert_block",
}
N_FEATURES_EXPECTED = int(features_full.EXPECTED_FEATURE_COUNTS["realized_total"])

key_sets = {}
result_summary = []
reference_metadata = None
reference_feature_columns = None
reference_shared_features = None

for out_name, method_name in DSP_METHODS.items():
    print("-" * 72)
    print(f"Building branch: {out_name} (method={method_name})")
    print("-" * 72)

    df = build_features(method_name)

    if df.empty:
        raise RuntimeError(f"Nhánh {out_name} không tạo được feature row nào.")

    # Kiểm tra method metadata do features_full ghi ra; không ghi đè alias.
    if "dsp_method" not in df.columns:
        raise AssertionError("features_full không xuất cột metadata dsp_method.")
    actual_methods = set(df["dsp_method"].dropna().astype(str).unique())
    if actual_methods != {method_name}:
        raise AssertionError(
            f"Nhánh {out_name}: dsp_method trong bảng là {actual_methods}, "
            f"không khớp method yêu cầu {method_name}."
        )

    # Mỗi cửa sổ chỉ được sinh đúng một lần.
    if df.duplicated(["file_id", "window_idx"]).any():
        duplicated = df.loc[
            df.duplicated(["file_id", "window_idx"], keep=False),
            ["file_id", "window_idx"],
        ].head()
        raise AssertionError(
            f"Nhánh {out_name} có cặp (file_id, window_idx) bị trùng:\n{duplicated}"
        )

    if (df["start_idx"] < WARMUP).any():
        bad = df.loc[df["start_idx"] < WARMUP, ["file_id", "window_idx", "start_idx"]].head()
        raise AssertionError(
            f"Nhánh {out_name} có cửa sổ nằm trong warm-up:\n{bad}"
        )

    # Xác nhận file_id là định danh file gốc, không phải định danh cửa sổ.
    if df["file_id"].isna().any():
        raise AssertionError(f"Nhánh {out_name} có file_id bị thiếu.")
    if not df.groupby("file_id")["label"].nunique().eq(1).all():
        raise AssertionError(f"Nhánh {out_name}: một file_id có nhiều label.")

    feature_cols = features_full.feature_columns(
        df,
        expected_count=N_FEATURES_EXPECTED,
        strict_prefix=True,
    )
    if reference_feature_columns is None:
        reference_feature_columns = list(feature_cols)
    elif list(feature_cols) != reference_feature_columns:
        raise AssertionError(
            f"Nhánh {out_name}: danh sách/thứ tự 28 feature không khớp nhánh đầu tiên."
        )

    # Time + Order phải giống hệt giữa các nhánh; chỉ Envelope được phép thay đổi.
    shared_feature_cols = [
        column for column in feature_cols
        if column.startswith(("time_", "order_"))
    ]
    expected_shared = features_full.EXPECTED_FEATURE_COUNTS["realized_time"] + \
        features_full.EXPECTED_FEATURE_COUNTS["realized_order"]
    if len(shared_feature_cols) != expected_shared:
        raise AssertionError(
            f"Kỳ vọng {expected_shared} feature Time+Order, "
            f"nhưng nhánh {out_name} có {len(shared_feature_cols)}."
        )
    current_shared = (
        df[["file_id", "window_idx", *shared_feature_cols]]
        .sort_values(["file_id", "window_idx"])
        .reset_index(drop=True)
    )
    if reference_shared_features is None:
        reference_shared_features = current_shared.copy()
        reference_shared_branch = out_name
    else:
        try:
            pd.testing.assert_frame_equal(
                reference_shared_features,
                current_shared,
                check_dtype=False,
                check_exact=False,
                rtol=1e-10,
                atol=1e-12,
            )
        except AssertionError as exc:
            raise AssertionError(
                f"Time/Order feature của {out_name} khác nhánh "
                f"{reference_shared_branch}; chỉ Envelope được phép khác. {exc}"
            ) from exc

    feature_values = df[feature_cols].to_numpy(dtype=np.float64)
    if np.isinf(feature_values).any():
        raise ValueError(f"Nhánh {out_name} chứa feature +Inf/-Inf.")
    n_nan = int(np.isnan(feature_values).sum())
    if n_nan:
        print(
            f"[CẢNH BÁO] Nhánh {out_name} có {n_nan} giá trị NaN trong feature; "
            "cần xử lý/điều tra trước khi huấn luyện MLP."
        )

    # Tập khóa cửa sổ và metadata phải giống hệt giữa các nhánh.
    keys = set(zip(df["file_id"].astype(str), df["window_idx"].astype(int)))
    key_sets[out_name] = keys

    metadata_cols = [
        "file_id", "window_idx", "start_idx", "label", "class_label",
        "load_hp", "fault_diameter_mils", "rpm", "rpm_source",
    ]
    missing_meta = [column for column in metadata_cols if column not in df.columns]
    if missing_meta:
        raise ValueError(f"Nhánh {out_name} thiếu metadata: {missing_meta}")

    current_metadata = (
        df[metadata_cols]
        .sort_values(["file_id", "window_idx"])
        .reset_index(drop=True)
    )
    if reference_metadata is None:
        reference_metadata = current_metadata.copy()
        reference_branch = out_name
    else:
        try:
            pd.testing.assert_frame_equal(
                reference_metadata,
                current_metadata,
                check_dtype=False,
                check_exact=False,
                rtol=0.0,
                atol=1e-12,
            )
        except AssertionError as exc:
            raise AssertionError(
                f"Metadata/window alignment của {out_name} không khớp "
                f"nhánh {reference_branch}: {exc}"
            ) from exc

    out_path = TABLES_DIR / f"features_mlp_{out_name}.parquet"
    df.to_parquet(out_path, index=False)

    win_per_file = df.groupby("file_id")["window_idx"].nunique()
    result_summary.append({
        "branch": out_name,
        "dsp_method": method_name,
        "n_rows": int(len(df)),
        "n_files": int(df["file_id"].nunique()),
        "n_features": int(len(feature_cols)),
        "min_windows_per_file": int(win_per_file.min()),
        "max_windows_per_file": int(win_per_file.max()),
        "n_nan_features": n_nan,
        "labels": df["label"].value_counts().to_dict(),
        "output": out_path.name,
    })

    print(
        f"OK {out_name}: {len(df)} windows / {df['file_id'].nunique()} files / "
        f"{len(feature_cols)} features -> {out_path.name}"
    )
    print(df["label"].value_counts().to_string())

# Kiểm tra chéo khóa cửa sổ sau khi đã build đủ ba nhánh.
reference_name = next(iter(key_sets))
reference_keys = key_sets[reference_name]
for branch_name, keys in key_sets.items():
    if keys != reference_keys:
        missing_here = list(reference_keys - keys)[:5]
        extra_here = list(keys - reference_keys)[:5]
        raise AssertionError(
            f"{branch_name} không thẳng hàng với {reference_name}. "
            f"Thiếu={missing_here}; thừa={extra_here}"
        )

print(
    f"OK: cả {len(key_sets)} nhánh có cùng {len(reference_keys)} "
    "(file_id, window_idx)."
)
print(
    "OK: 21 feature Time+Order giống nhau giữa các nhánh; "
    "7 feature Envelope được phép khác nhau."
)


------------------------------------------------------------------------
Building branch: hybrid (method=hybrid)
------------------------------------------------------------------------
OK hybrid: 4563 windows / 40 files / 28 features -> features_mlp_hybrid.parquet
label
OR        1389
B         1387
IR        1386
Normal     401
------------------------------------------------------------------------
Building branch: hilbert (method=hilbert_offline)
------------------------------------------------------------------------
OK hilbert: 4563 windows / 40 files / 28 features -> features_mlp_hilbert.parquet
label
OR        1389
B         1387
IR        1386
Normal     401
------------------------------------------------------------------------
Building branch: hilbert_block (method=hilbert_block)
------------------------------------------------------------------------
OK hilbert_block: 4563 windows / 40 files / 28 features -> features_mlp_hilbert_block.parquet
label
OR        1389
B        

## 5. Tạo alias Hybrid và kiểm tra kết quả


In [7]:
import shutil

# Alias chính cho huấn luyện/đánh giá mặc định vẫn là Hybrid.
hybrid_path = TABLES_DIR / "features_mlp_hybrid.parquet"
alias_path = TABLES_DIR / "features_mlp.parquet"
if not hybrid_path.exists():
    raise FileNotFoundError(f"Chưa có bảng Hybrid: {hybrid_path}")
shutil.copy2(hybrid_path, alias_path)
print("Alias: features_mlp.parquet <- features_mlp_hybrid.parquet")

print("=" * 88)
print("TÓM TẮT BA NHÁNH FEATURE")
print("=" * 88)
for result in result_summary:
    print(
        f"{result['branch']:14s} | {result['n_rows']:>6} windows | "
        f"{result['n_files']:>3} files | {result['n_features']} features | "
        f"{result['min_windows_per_file']}-{result['max_windows_per_file']} "
        f"windows/file | NaN={result['n_nan_features']} | {result['output']}"
    )
print("=" * 88)

df_check = pd.read_parquet(alias_path)
display(df_check.head(5))

# Xuất CSV cho cả ba nhánh để kiểm tra thủ công; Parquet vẫn là nguồn chính.
for out_name in DSP_METHODS:
    source_path = TABLES_DIR / f"features_mlp_{out_name}.parquet"
    csv_path = TABLES_DIR / f"features_mlp_{out_name}.csv"
    pd.read_parquet(source_path).to_csv(csv_path, index=False)
    print(f"Đã xuất {csv_path.name} ({csv_path.stat().st_size / 1024:.1f} KB)")

# Alias CSV của Hybrid.
pd.read_parquet(alias_path).to_csv(TABLES_DIR / "features_mlp.csv", index=False)
print("Đã xuất features_mlp.csv (alias Hybrid).")

meta_present = [column for column in features_full.METADATA_COLUMNS if column in df_check.columns]
feature_cols_check = features_full.feature_columns(
    df_check,
    expected_count=N_FEATURES_EXPECTED,
    strict_prefix=True,
)
print(f"Metadata: {meta_present}")
print(f"Số feature: {len(feature_cols_check)}")
print(
    f"Số file: {df_check['file_id'].nunique()} | "
    f"Số cửa sổ: {len(df_check)}"
)
print(
    f"Ví dụ file_id: {df_check['file_id'].iloc[0]} "
    "(phải là tên file gốc, không chứa chỉ số cửa sổ)."
)

Alias: features_mlp.parquet <- features_mlp_hybrid.parquet
TÓM TẮT BA NHÁNH FEATURE
hybrid         |   4563 windows |  40 files | 28 features | 56-117 windows/file | NaN=0 | features_mlp_hybrid.parquet
hilbert        |   4563 windows |  40 files | 28 features | 56-117 windows/file | NaN=0 | features_mlp_hilbert.parquet
hilbert_block  |   4563 windows |  40 files | 28 features | 56-117 windows/file | NaN=0 | features_mlp_hilbert_block.parquet


,file_id,window_idx,start_idx,label,class_label,load_hp,fault_diameter_mils,rpm,rpm_source,source_sample_rate_hz,...,order_BPFO_h2_and_BSF_h3,order_BPFO_h3_and_BPFI_h2,order_BPFI_h3,envelope_BSF_h1,envelope_BPFO_h1,envelope_BSF_h2,envelope_BPFI_h1,envelope_BPFO_h2_and_BSF_h3,envelope_BPFO_h3_and_BPFI_h2,envelope_BPFI_h3
0,118_0,2,2048,B,B_007,0,7.0,1796.0,measured,12000.0,...,0.000884,0.001728,0.003051,0.003963,0.014368,0.017429,0.013649,0.023849,0.006226,0.011753
1,118_0,3,3072,B,B_007,0,7.0,1796.0,measured,12000.0,...,0.000917,0.001172,0.000607,0.021818,0.021108,0.004818,0.008866,0.007195,0.013521,0.012865
2,118_0,4,4096,B,B_007,0,7.0,1796.0,measured,12000.0,...,0.000822,0.000914,0.001095,0.028146,0.009952,0.007426,0.015324,0.005727,0.007093,0.009974
3,118_0,5,5120,B,B_007,0,7.0,1796.0,measured,12000.0,...,0.001183,0.001212,0.001244,0.015476,0.008083,0.012558,0.022247,0.025641,0.010151,0.009193
4,118_0,6,6144,B,B_007,0,7.0,1796.0,measured,12000.0,...,0.001257,0.000829,0.001144,0.013502,0.007322,0.015021,0.006337,0.026316,0.003489,0.007988


Đã xuất features_mlp_hybrid.csv (2966.1 KB)
Đã xuất features_mlp_hilbert.csv (3006.5 KB)
Đã xuất features_mlp_hilbert_block.csv (2997.7 KB)
Đã xuất features_mlp.csv (alias Hybrid).
Metadata: ['file_id', 'window_idx', 'start_idx', 'label', 'class_label', 'load_hp', 'fault_diameter_mils', 'rpm', 'rpm_source', 'source_sample_rate_hz', 'source_sample_rate_source', 'feature_sample_rate_hz', 'dsp_method']
Số feature: 28
Số file: 40 | Số cửa sổ: 4563
Ví dụ file_id: 118_0 (phải là tên file gốc, không chứa chỉ số cửa sổ).


In [8]:
# Kiểm tra hợp đồng feature trên bảng alias Hybrid (28 chiều).
cols = features_full.feature_columns(
    df_check,
    expected_count=features_full.EXPECTED_FEATURE_COUNTS["realized_total"],
    strict_prefix=True,
)
print(f"Số feature hợp lệ: {len(cols)}")

Số feature hợp lệ: 28


In [9]:
print("10 feature đầu:", cols[:10])
print("10 feature cuối:", cols[-10:])

10 feature đầu: ['time_mean', 'time_std', 'time_rms', 'time_peak', 'time_kurtosis', 'time_skewness', 'time_variance', 'time_crest_factor', 'time_shape_factor', 'time_impulse_factor']
10 feature cuối: ['order_BPFO_h2_and_BSF_h3', 'order_BPFO_h3_and_BPFI_h2', 'order_BPFI_h3', 'envelope_BSF_h1', 'envelope_BPFO_h1', 'envelope_BSF_h2', 'envelope_BPFI_h1', 'envelope_BPFO_h2_and_BSF_h3', 'envelope_BPFO_h3_and_BPFI_h2', 'envelope_BPFI_h3']
